# NB_bishop_ch08_figures

**Bishop Chapter 8 — Key Figures: Computational Graphs, Chain Rule & Backpropagation Diagrams**

This notebook generates publication-quality figures for Bishop Chapter 8: computational graph diagrams, forward/backward pass illustrations, and a numerical backpropagation example with annotated gradients.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_08/NB_bishop_ch08_figures.ipynb)

In [ ]:
# Install dependencies (Colab-safe)
# !pip install numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

print(f'NumPy version: {np.__version__}')

In [ ]:
# Chart styling setup
COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

def save_fig(fig, name, chart_dir='../../charts'):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True, dpi=300)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=300)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

## Figure 8.1: Computational Graph for $f(x, y) = (x + y) \cdot y$

We decompose the function into elementary operations and draw the directed acyclic graph.

In [ ]:
def draw_rounded_box(ax, xy, text, width=1.2, height=0.7, color=COLORS['blue'],
                     fontsize=13, text_color='white'):
    """Draw a rounded rectangle node."""
    x, y = xy
    box = FancyBboxPatch((x - width/2, y - height/2), width, height,
                         boxstyle='round,pad=0.1', facecolor=color,
                         edgecolor='none', alpha=0.9, zorder=5)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            color=text_color, fontweight='bold', zorder=6)
    return (x, y)


def draw_arrow(ax, start, end, color=COLORS['blue'], lw=1.5, label=None,
               label_offset=(0, 0.2), fontsize=10, label_color=None):
    """Draw an arrow between two points with optional label."""
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=lw,
                                shrinkA=8, shrinkB=8))
    if label:
        mx = (start[0] + end[0]) / 2 + label_offset[0]
        my = (start[1] + end[1]) / 2 + label_offset[1]
        lc = label_color if label_color else color
        ax.text(mx, my, label, fontsize=fontsize, ha='center', va='center',
                color=lc)


# Computational graph: f(x,y) = (x+y) * y
# Nodes: x, y, q = x+y, f = q*y
# With numerical values: x=2, y=3 -> q=5, f=15

fig, ax = plt.subplots(figsize=(12, 6))

# Input nodes
draw_rounded_box(ax, (1, 4), '$x = 2$', color=COLORS['blue'])
draw_rounded_box(ax, (1, 1.5), '$y = 3$', color=COLORS['blue'])

# Addition node
draw_rounded_box(ax, (4.5, 3.2), '$+$', width=0.9, height=0.9,
                 color=COLORS['green'])

# Multiplication node
draw_rounded_box(ax, (8, 2.5), '$\\times$', width=0.9, height=0.9,
                 color=COLORS['amber'])

# Output node
draw_rounded_box(ax, (11, 2.5), '$f = 15$', color=COLORS['red'])

# Forward arrows with values
draw_arrow(ax, (1, 4), (4.5, 3.2), color=COLORS['blue'], lw=2,
           label='$x=2$', label_offset=(-0.3, 0.4), fontsize=11,
           label_color='#333333')
draw_arrow(ax, (1, 1.5), (4.5, 3.2), color=COLORS['blue'], lw=2,
           label='$y=3$', label_offset=(-0.5, 0.3), fontsize=11,
           label_color='#333333')
draw_arrow(ax, (4.5, 3.2), (8, 2.5), color=COLORS['green'], lw=2,
           label='$q = x+y = 5$', label_offset=(0, 0.5), fontsize=11,
           label_color='#333333')
draw_arrow(ax, (1, 1.5), (8, 2.5), color=COLORS['blue'], lw=2,
           label='$y=3$', label_offset=(0.3, -0.5), fontsize=11,
           label_color='#333333')
draw_arrow(ax, (8, 2.5), (11, 2.5), color=COLORS['amber'], lw=2,
           label='$f = q \\cdot y = 15$', label_offset=(0, 0.5), fontsize=11,
           label_color='#333333')

ax.set_xlim(-0.5, 13)
ax.set_ylim(0, 5.5)
ax.set_title('Figure 8.1: Computational Graph for $f(x, y) = (x + y) \\cdot y$',
             fontsize=14, pad=15)
ax.text(6, 0.3, 'Forward pass: compute values left to right',
        fontsize=11, ha='center', color=COLORS['blue'], style='italic')
ax.axis('off')
fig.tight_layout()
save_fig(fig, 'fig8_1_computational_graph')
plt.show()

## Figure 8.2: Chain Rule — Forward and Backward Passes

Diagram showing how the chain rule propagates gradients backward through the same computational graph. Each node shows the local gradient and accumulated upstream gradient.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# --- Top panel: Forward pass ---
ax = axes[0]
nodes_fwd = {
    'x': (1, 2), 'y': (1, 0.5),
    '+': (4, 1.5), '*': (7, 1.0), 'f': (10, 1.0)
}

draw_rounded_box(ax, nodes_fwd['x'], '$x = 2$', color=COLORS['blue'], fontsize=11)
draw_rounded_box(ax, nodes_fwd['y'], '$y = 3$', color=COLORS['blue'], fontsize=11)
draw_rounded_box(ax, nodes_fwd['+'], '$+$', width=0.8, height=0.8, color=COLORS['green'])
draw_rounded_box(ax, nodes_fwd['*'], '$\\times$', width=0.8, height=0.8, color=COLORS['amber'])
draw_rounded_box(ax, nodes_fwd['f'], '$f=15$', color=COLORS['red'], fontsize=11)

draw_arrow(ax, nodes_fwd['x'], nodes_fwd['+'], color=COLORS['blue'], lw=2,
           label='2', label_offset=(-0.3, 0.3))
draw_arrow(ax, nodes_fwd['y'], nodes_fwd['+'], color=COLORS['blue'], lw=2,
           label='3', label_offset=(-0.3, 0.15))
draw_arrow(ax, nodes_fwd['+'], nodes_fwd['*'], color=COLORS['green'], lw=2,
           label='$q=5$', label_offset=(0, 0.4))
draw_arrow(ax, nodes_fwd['y'], nodes_fwd['*'], color=COLORS['blue'], lw=2,
           label='3', label_offset=(0, -0.35))
draw_arrow(ax, nodes_fwd['*'], nodes_fwd['f'], color=COLORS['amber'], lw=2,
           label='15', label_offset=(0, 0.35))

ax.set_title('Forward Pass: Compute Values Left $\\rightarrow$ Right',
             fontsize=13, pad=10, color=COLORS['blue'])
ax.set_xlim(-0.5, 12)
ax.set_ylim(-0.5, 3.5)
ax.axis('off')

# --- Bottom panel: Backward pass ---
ax = axes[1]

nodes_bwd = {
    'x': (1, 2), 'y': (1, 0.5),
    '+': (4, 1.5), '*': (7, 1.0), 'f': (10, 1.0)
}

# Draw nodes with gradient annotations
draw_rounded_box(ax, nodes_bwd['x'], '$x$', color='#AAAAAA', fontsize=11, text_color='#333')
draw_rounded_box(ax, nodes_bwd['y'], '$y$', color='#AAAAAA', fontsize=11, text_color='#333')
draw_rounded_box(ax, nodes_bwd['+'], '$+$', width=0.8, height=0.8,
                 color='#AAAAAA', text_color='#333')
draw_rounded_box(ax, nodes_bwd['*'], '$\\times$', width=0.8, height=0.8,
                 color='#AAAAAA', text_color='#333')
draw_rounded_box(ax, nodes_bwd['f'], '$f$', color='#AAAAAA', fontsize=11, text_color='#333')

# Backward arrows (right to left) with gradient values
# df/df = 1
draw_arrow(ax, nodes_bwd['f'], nodes_bwd['*'], color=COLORS['red'], lw=2.5,
           label='$\\frac{\\partial f}{\\partial f} = 1$',
           label_offset=(0, 0.55), fontsize=11, label_color=COLORS['red'])

# df/dq = y = 3,  df/dy (via *) = q = 5
draw_arrow(ax, nodes_bwd['*'], nodes_bwd['+'], color=COLORS['red'], lw=2.5,
           label='$\\frac{\\partial f}{\\partial q} = y = 3$',
           label_offset=(0, 0.55), fontsize=11, label_color=COLORS['red'])
draw_arrow(ax, nodes_bwd['*'], nodes_bwd['y'], color=COLORS['red'], lw=2.5,
           label='$\\frac{\\partial f}{\\partial y}\\big|_* = q = 5$',
           label_offset=(0.5, -0.45), fontsize=10, label_color=COLORS['red'])

# Through +: df/dx = df/dq * dq/dx = 3*1 = 3
draw_arrow(ax, nodes_bwd['+'], nodes_bwd['x'], color=COLORS['red'], lw=2.5,
           label='$\\frac{\\partial f}{\\partial x} = 3 \\times 1 = 3$',
           label_offset=(-0.5, 0.35), fontsize=10, label_color=COLORS['red'])

# df/dy via + : df/dq * dq/dy = 3*1 = 3, total df/dy = 5 + 3 = 8
draw_arrow(ax, nodes_bwd['+'], nodes_bwd['y'], color=COLORS['red'], lw=2.5,
           label='$\\frac{\\partial f}{\\partial y}\\big|_+ = 3 \\times 1 = 3$',
           label_offset=(-0.8, 0.15), fontsize=10, label_color=COLORS['red'])

# Summary box
ax.text(6, 3.0,
        'Total: $\\frac{\\partial f}{\\partial x} = 3$, '
        '$\\frac{\\partial f}{\\partial y} = 5 + 3 = 8$',
        fontsize=13, ha='center', va='center', color=COLORS['red'],
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor=COLORS['red'], alpha=0.9))

ax.set_title('Backward Pass: Propagate Gradients Right $\\leftarrow$ Left (Chain Rule)',
             fontsize=13, pad=10, color=COLORS['red'])
ax.set_xlim(-0.5, 12)
ax.set_ylim(-0.8, 3.8)
ax.axis('off')

fig.suptitle('Figure 8.2: Forward and Backward Passes Through a Computational Graph',
             fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig8_2_chain_rule')
plt.show()

## Figure 8.3: Numerical Backpropagation Through a 2-Layer Network

A complete annotated diagram showing forward values (blue) and backward gradients (red) through a small 2-layer network with sigmoid activation.

**Network**: input $x = [0.5]$, hidden layer with 2 units (sigmoid), linear output, MSE loss. Target $y = 1.0$.

In [ ]:
# Numerical backprop computation
# Architecture: 1 -> 2 (sigmoid) -> 1 (linear), MSE loss

# Parameters
x_in = np.array([0.5])
y_true = np.array([1.0])

w1 = np.array([[0.8, -0.5]])   # (1, 2)
b1 = np.array([0.1, 0.2])     # (2,)
w2 = np.array([[0.6], [0.9]]) # (2, 1)
b2 = np.array([0.1])          # (1,)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Forward pass
z1 = x_in @ w1 + b1       # (1, 2)
h1 = sigmoid(z1)           # (1, 2)
z2 = h1 @ w2 + b2         # (1, 1)
y_pred = z2                # linear output
loss = 0.5 * (y_pred - y_true)**2

print('=== FORWARD PASS ===')
print(f'x     = {x_in}')
print(f'z1    = x*w1 + b1 = {z1}')
print(f'h1    = sigma(z1) = {h1}')
print(f'z2    = h1*w2 + b2 = {z2}')
print(f'y_hat = {y_pred}')
print(f'loss  = 0.5*(y_hat - y)^2 = {loss}')

# Backward pass
dL_dy = y_pred - y_true         # (1, 1)
dL_dz2 = dL_dy                  # linear activation
dL_dw2 = h1.T @ dL_dz2         # (2, 1)
dL_db2 = dL_dz2.flatten()      # (1,)
dL_dh1 = dL_dz2 @ w2.T         # (1, 2)
sig_deriv = h1 * (1 - h1)      # (1, 2)
dL_dz1 = dL_dh1 * sig_deriv    # (1, 2)
dL_dw1 = x_in.reshape(-1, 1) @ dL_dz1  # (1, 2)
dL_db1 = dL_dz1.flatten()      # (2,)

print('\n=== BACKWARD PASS ===')
print(f'dL/dy_hat = {dL_dy}')
print(f'dL/dz2    = {dL_dz2}')
print(f'dL/dw2    = {dL_dw2.flatten()}')
print(f'dL/db2    = {dL_db2}')
print(f'dL/dh1    = {dL_dh1}')
print(f'sig_deriv = {sig_deriv}')
print(f'dL/dz1    = {dL_dz1}')
print(f'dL/dw1    = {dL_dw1}')
print(f'dL/db1    = {dL_db1}')

In [ ]:
# Draw the annotated backpropagation diagram
fig, ax = plt.subplots(figsize=(16, 8))

# Layout: 6 columns
#   x -> z1 -> h1=sigma(z1) -> z2 -> y_hat -> L
col_x = 1.0
col_z1 = 4.0
col_h1 = 7.0
col_z2 = 10.0
col_y = 13.0
col_L = 15.5

row_top = 5.0
row_bot = 2.5

# Draw nodes
draw_rounded_box(ax, (col_x, 3.75), f'$x = {x_in[0]}$',
                 color=COLORS['blue'], fontsize=11, width=1.5)

# z1 nodes (two hidden units)
draw_rounded_box(ax, (col_z1, row_top),
                 f'$z_1^{{(1)}} = {z1[0,0]:.3f}$',
                 color=COLORS['green'], fontsize=9, width=1.8)
draw_rounded_box(ax, (col_z1, row_bot),
                 f'$z_1^{{(2)}} = {z1[0,1]:.3f}$',
                 color=COLORS['green'], fontsize=9, width=1.8)

# h1 nodes
draw_rounded_box(ax, (col_h1, row_top),
                 f'$h_1^{{(1)}} = {h1[0,0]:.4f}$',
                 color=COLORS['green'], fontsize=9, width=1.8)
draw_rounded_box(ax, (col_h1, row_bot),
                 f'$h_1^{{(2)}} = {h1[0,1]:.4f}$',
                 color=COLORS['green'], fontsize=9, width=1.8)

# z2, y_hat, L
draw_rounded_box(ax, (col_z2, 3.75),
                 f'$z_2 = {z2[0,0]:.4f}$',
                 color=COLORS['amber'], fontsize=10, width=1.8)
draw_rounded_box(ax, (col_y, 3.75),
                 f'$\\hat{{y}} = {y_pred[0,0]:.4f}$',
                 color=COLORS['amber'], fontsize=10, width=1.8)
draw_rounded_box(ax, (col_L, 3.75),
                 f'$L = {loss[0,0]:.4f}$',
                 color=COLORS['red'], fontsize=10, width=1.5)

# Forward arrows (blue)
# x -> z1 (two arrows)
draw_arrow(ax, (col_x, 3.75), (col_z1, row_top), color=COLORS['blue'], lw=1.5,
           label=f'$w_1^{{(1)}}={w1[0,0]}$', label_offset=(-0.6, 0.4), fontsize=9,
           label_color=COLORS['blue'])
draw_arrow(ax, (col_x, 3.75), (col_z1, row_bot), color=COLORS['blue'], lw=1.5,
           label=f'$w_1^{{(2)}}={w1[0,1]}$', label_offset=(-0.6, -0.3), fontsize=9,
           label_color=COLORS['blue'])

# z1 -> h1 (sigma)
draw_arrow(ax, (col_z1, row_top), (col_h1, row_top), color=COLORS['green'], lw=1.5,
           label='$\\sigma$', label_offset=(0, 0.35), fontsize=10,
           label_color=COLORS['green'])
draw_arrow(ax, (col_z1, row_bot), (col_h1, row_bot), color=COLORS['green'], lw=1.5,
           label='$\\sigma$', label_offset=(0, 0.35), fontsize=10,
           label_color=COLORS['green'])

# h1 -> z2
draw_arrow(ax, (col_h1, row_top), (col_z2, 3.75), color=COLORS['amber'], lw=1.5,
           label=f'$w_2^{{(1)}}={w2[0,0]}$', label_offset=(-0.3, 0.4), fontsize=9,
           label_color=COLORS['amber'])
draw_arrow(ax, (col_h1, row_bot), (col_z2, 3.75), color=COLORS['amber'], lw=1.5,
           label=f'$w_2^{{(2)}}={w2[1,0]}$', label_offset=(-0.3, -0.3), fontsize=9,
           label_color=COLORS['amber'])

# z2 -> y_hat -> L
draw_arrow(ax, (col_z2, 3.75), (col_y, 3.75), color=COLORS['amber'], lw=1.5,
           label='linear', label_offset=(0, 0.35), fontsize=9,
           label_color='#666666')
draw_arrow(ax, (col_y, 3.75), (col_L, 3.75), color=COLORS['red'], lw=1.5,
           label='MSE', label_offset=(0, 0.35), fontsize=9,
           label_color=COLORS['red'])

# Backward gradient annotations (red, below the arrows)
grad_y = 1.2

# Gradient values
ax.text(col_L, grad_y, f'$\\frac{{\\partial L}}{{\\partial L}} = 1$',
        fontsize=10, ha='center', color=COLORS['red'])
ax.text(col_y, grad_y, f'$\\frac{{\\partial L}}{{\\partial \\hat{{y}}}} = {dL_dy[0,0]:.4f}$',
        fontsize=10, ha='center', color=COLORS['red'])
ax.text(col_z2, grad_y, f'$\\frac{{\\partial L}}{{\\partial z_2}} = {dL_dz2[0,0]:.4f}$',
        fontsize=10, ha='center', color=COLORS['red'])
ax.text(col_h1, grad_y + 0.6,
        f'$\\frac{{\\partial L}}{{\\partial h_1^{{(1)}}}} = {dL_dh1[0,0]:.4f}$',
        fontsize=9, ha='center', color=COLORS['red'])
ax.text(col_h1, grad_y - 0.3,
        f'$\\frac{{\\partial L}}{{\\partial h_1^{{(2)}}}} = {dL_dh1[0,1]:.4f}$',
        fontsize=9, ha='center', color=COLORS['red'])
ax.text(col_z1, grad_y + 0.6,
        f'$\\frac{{\\partial L}}{{\\partial z_1^{{(1)}}}} = {dL_dz1[0,0]:.4f}$',
        fontsize=9, ha='center', color=COLORS['red'])
ax.text(col_z1, grad_y - 0.3,
        f'$\\frac{{\\partial L}}{{\\partial z_1^{{(2)}}}} = {dL_dz1[0,1]:.4f}$',
        fontsize=9, ha='center', color=COLORS['red'])

# Weight gradient summary box
summary = (
    f'$\\frac{{\\partial L}}{{\\partial w_1}} = [{dL_dw1[0,0]:.4f},\\; {dL_dw1[0,1]:.4f}]$\n'
    f'$\\frac{{\\partial L}}{{\\partial w_2}} = [{dL_dw2[0,0]:.4f},\\; {dL_dw2[1,0]:.4f}]$'
)
ax.text(8.5, 0.3, summary, fontsize=11, ha='center', va='center',
        color=COLORS['red'],
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor=COLORS['red'], alpha=0.9))

# Labels
ax.text(8.5, 6.5,
        'Forward values (top, in nodes) | Backward gradients (bottom, in red)',
        fontsize=12, ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#F0F0F0',
                  edgecolor='#999999', alpha=0.8))

ax.set_xlim(-0.5, 17)
ax.set_ylim(-0.5, 7)
ax.set_title('Numerical Backpropagation Through a 2-Layer Network',
             fontsize=14, pad=15)
ax.axis('off')
fig.tight_layout()
save_fig(fig, 'fig8_backprop_example')
plt.show()

In [ ]:
# Verify with finite differences
eps = 1e-5

def compute_loss(w1_, b1_, w2_, b2_):
    z1_ = x_in @ w1_ + b1_
    h1_ = sigmoid(z1_)
    z2_ = h1_ @ w2_ + b2_
    return 0.5 * np.sum((z2_ - y_true)**2)

# Numerical gradient for w1
dL_dw1_num = np.zeros_like(w1)
for i in range(w1.shape[0]):
    for j in range(w1.shape[1]):
        w1p = w1.copy(); w1p[i, j] += eps
        w1m = w1.copy(); w1m[i, j] -= eps
        dL_dw1_num[i, j] = (compute_loss(w1p, b1, w2, b2) -
                            compute_loss(w1m, b1, w2, b2)) / (2 * eps)

dL_dw2_num = np.zeros_like(w2)
for i in range(w2.shape[0]):
    for j in range(w2.shape[1]):
        w2p = w2.copy(); w2p[i, j] += eps
        w2m = w2.copy(); w2m[i, j] -= eps
        dL_dw2_num[i, j] = (compute_loss(w1, b1, w2p, b2) -
                            compute_loss(w1, b1, w2m, b2)) / (2 * eps)

print('=== GRADIENT VERIFICATION (finite differences) ===')
print(f'dL/dw1 analytical: {dL_dw1.flatten()}')
print(f'dL/dw1 numerical:  {dL_dw1_num.flatten()}')
print(f'Max diff: {np.max(np.abs(dL_dw1 - dL_dw1_num)):.2e}')
print(f'\ndL/dw2 analytical: {dL_dw2.flatten()}')
print(f'dL/dw2 numerical:  {dL_dw2_num.flatten()}')
print(f'Max diff: {np.max(np.abs(dL_dw2 - dL_dw2_num)):.2e}')

In [ ]:
print('Notebook complete.')